In [ ]:
# ---------- Importe ----------
import json # zum Speichern der Daten als JSON-Datei
import os # für Ordner und Dateipfade
import pandas as pd # für die Tabellen

In [ ]:
# Überprüfung Semaglutid Juli 2025
df = pd.read_csv('data/raw/berichte_pro_tag.csv', parse_dates=['time'])  # CSV-Datei einlesen und die Spalte "time" als Datum interpretieren
df.head()

In [ ]:
df.dtypes

In [ ]:
semaglutide_mask = df['wirkstoff'] == 'semaglutide'
mask_juni = (df['time'] >= '2025-06-01') & (df['time'] <= '2025-06-30')
df[semaglutide_mask & mask_juni].groupby('time')['count'].sum() # brauche eigentlich nicht zu gruppieren, aber mir gefällt die Ansicht besser

In [ ]:
semaglutide_mask = df['wirkstoff'] == 'semaglutide'
mask_juli = (df['time'] >= '2025-07-01') & (df['time'] <= '2025-07-31')
df[semaglutide_mask & mask_juli].groupby('time')['count'].sum()

In [ ]:
with open('data/raw/semaglutide_berichte_20250701_bis_20250731.json', 'r') as f:
    berichte = json.load(f)

In [ ]:
len(berichte)

In [ ]:
df_juli = pd.DataFrame()   # leere Tabelle anlegen

for i in range(len(berichte)):   # für jeden Bericht (wie range(df.shape[0]))
    bericht_dict = berichte[i]   # ein Bericht als Dictionary (wie response_dict)
    quelle = bericht_dict.get('primarysource') or {}   # Infos zur meldenden Person, leer falls nicht vorhanden

    df_juli.loc[i, 'id'] = bericht_dict['safetyreportid']   # eindeutige Nummer des Berichts
    df_juli.loc[i, 'datum'] = bericht_dict['receivedate']    # Tag an dem FDA Bericht erhalten hat
    df_juli.loc[i, 'meldende'] = quelle.get('qualification') # Wer den Fall urspünglich gemeldet hat (z.b. 1= Arzt/Ärztin)
    df_juli.loc[i, 'land'] = bericht_dict.get('occurcountry') # Land, in dem die Nebenwirkungen aufgetreten sind
    df_juli.loc[i, 'berichtsart'] = bericht_dict.get('reporttype') # Woher der Bericht stammt (z.B. 1 = Spontanmeldung)
    df_juli.loc[i, 'firmennummer'] = bericht_dict.get('companynumb') # Fallnummer des Herstellers, der den Fall weitergeleitet hat

df_juli.head()

In [ ]:
mask_ausreisser = df_juli['datum'].isin(['20250715', '20250716', '20250718'])
df_juli[mask_ausreisser]['meldende'].value_counts(normalize=True)

In [ ]:
df_juli[~mask_ausreisser]['meldende'].value_counts(normalize=True)

In [ ]:
mask_ausreisser = df_juli['datum'].isin(['20250715', '20250716', '20250718'])
df_juli[mask_ausreisser]['land'].value_counts(normalize=True)

In [ ]:
df_juli[~mask_ausreisser]['land'].value_counts(normalize=True)

In [ ]:
mask_ausreisser = df_juli['datum'].isin(['20250715', '20250716', '20250718'])
df_juli[mask_ausreisser]['berichtsart'].value_counts(normalize=True)

In [ ]:
df_juli[~mask_ausreisser]['berichtsart'].value_counts(normalize=True)

In [ ]:
df_juli['firma'] = df_juli['firmennummer'].str.split('-').str[1]   # Firmenkürzel aus der Firmennummer herausziehen
df_juli[mask_ausreisser]['firma'].value_counts(normalize=True).head(10)    # Firmen an den Ausreißer-Tagen (Anteile, Top 10)

In [ ]:
df_juli[~mask_ausreisser]['firma'].value_counts(normalize=True).head(10)   # Firmen an den übrigen Tagen (Anteile, Top 10)

In [ ]:
ordner = 'data/raw'   # hier liegen die Rohdaten

# ---------- Alle JSON-Dateien im Ordner finden ----------
dateien = []   # leere Liste für die Dateinamen
for name in os.listdir(ordner):   # alle Dateien im Ordner durchgehen
    if name.endswith('.json'):    # nur JSON-Dateien (die CSV der Zeitreihe nicht)
        dateien.append(name)      # Dateinamen merken
print(len(dateien), 'Dateien gefunden')   # sollten 24 sein

In [ ]:
# ---------- Leere Listen für die drei Tabellen ----------
berichte_zeilen = []         # eine Zeile pro Bericht
nebenwirkungen_zeilen = []   # eine Zeile pro gemeldeter Nebenwirkung
medikamente_zeilen = []      # eine Zeile pro genanntem Medikament

# ---------- Durch alle Dateien und Berichte gehen ----------
for name in dateien:   # jede Monatsdatei
    abfrage = name.split('_')[0]   # aus 'semaglutide_berichte_...' wird 'semaglutide'
    with open(os.path.join(ordner, name), 'r') as f:   # Datei öffnen (os.path.join setzt Ordner + Name zusammen)
        berichte = json.load(f)   # Liste mit allen Berichten dieses Monats

    for bericht in berichte:   # jeden Bericht durchgehen
        bericht_id = bericht.get('safetyreportid')   # ID merken, sie verbindet später die drei Tabellen
        patient = bericht.get('patient') or {}       # Teil mit Patientendaten (leer, falls er fehlt)
        quelle = bericht.get('primarysource') or {}  # Teil mit der meldenden Person (leer, falls er fehlt)

        # --- Tabelle 1: Berichte ---
        berichte_zeilen.append({
            'id': bericht_id,
            'abfrage': abfrage,                                  # aus welchem Download der Bericht stammt
            'datum': bericht.get('receivedate'),                 # Eingangsdatum
            'schwerwiegend': bericht.get('serious'),             # 1 = ja, 2 = nein
            'geschlecht': patient.get('patientsex'),             # 0 = unbekannt, 1 = männlich, 2 = weiblich
            'alter': patient.get('patientonsetage'),             # Alter (Zahl)
            'alter_einheit': patient.get('patientonsetageunit'), # Einheit des Alters (Code)
            'meldende': quelle.get('qualification'),             # wer gemeldet hat
            'land': bericht.get('occurcountry'),                 # Land
            'berichtsart': bericht.get('reporttype'),            # Art des Berichts
            'firmennummer': bericht.get('companynumb'),          # Fallnummer des Herstellers
        })

# --- Tabelle 2: Nebenwirkungen ---
        for reaktion in patient.get('reaction', []):   # jede Nebenwirkung im Bericht (leere Liste, falls keine)
            nebenwirkungen_zeilen.append({
                'id': bericht_id,
                'abfrage': abfrage,                                 # aus welchem Download die Zeile stammt
                'nebenwirkung': reaktion.get('reactionmeddrapt'),   # Name der Nebenwirkung
            })

        # --- Tabelle 3: Medikamente ---
        for medikament in patient.get('drug', []):   # jedes Medikament im Bericht
            wirkstoff_info = medikament.get('activesubstance') or {}   # verschachtelter Teil mit dem Wirkstoff
            medikamente_zeilen.append({
                'id': bericht_id,
                'abfrage': abfrage,                                         # aus welchem Download die Zeile stammt
                'medikament': medikament.get('medicinalproduct'),          # eingetragener Name
                'wirkstoff': wirkstoff_info.get('activesubstancename'),     # Wirkstoff
                'rolle': medikament.get('drugcharacterization'),            # 1 = verdächtig, 2 = Begleitmedikation, 3 = Wechselwirkung
                'anwendung': medikament.get('drugindication'),              # wofür es genommen wurde
            })

# ---------- Aus den Listen Tabellen machen ----------
df_berichte = pd.DataFrame(berichte_zeilen)
df_nebenwirkungen = pd.DataFrame(nebenwirkungen_zeilen)
df_medikamente = pd.DataFrame(medikamente_zeilen)

print('Berichte:', df_berichte.shape)             # (Zeilen, Spalten)
print('Nebenwirkungen:', df_nebenwirkungen.shape)
print('Medikamente:', df_medikamente.shape)

In [ ]:
# ---------- Speichern ----------
os.makedirs('data/processed', exist_ok=True)   # Ordner anlegen, falls es ihn noch nicht gibt
df_berichte.to_csv('data/processed/berichte.csv', index=False)
df_nebenwirkungen.to_csv('data/processed/nebenwirkungen.csv', index=False)
df_medikamente.to_csv('data/processed/medikamente.csv', index=False)

In [ ]:
# Berichte können doppelt vorkommen, wenn Tirzepatid und Semagltuig gegeben wurde... wie oft kommt das vor?
df_berichte['id'].duplicated().sum()   # wie oft kommt eine ID mehr als einmal vor?

In [ ]:
# ---------- Duplikate entfernen ----------
mask_doppelt = df_berichte['id'].duplicated()        # Maske: True, wenn die ID schon weiter oben vorkam
doppelte_ids = df_berichte[mask_doppelt]['id']       # aus diesen Zeilen nur die Spalte id nehmen

mask_berichte = df_berichte['id'].isin(doppelte_ids) & (df_berichte['abfrage'] == 'tirzepatide')   # doppelte Zeilen aus dem Tirzepatid-Download
df_berichte = df_berichte.drop(df_berichte[mask_berichte].index)   # diese Zeilen entfernen

mask_nebenwirkungen = df_nebenwirkungen['id'].isin(doppelte_ids) & (df_nebenwirkungen['abfrage'] == 'tirzepatide')
df_nebenwirkungen = df_nebenwirkungen.drop(df_nebenwirkungen[mask_nebenwirkungen].index)

mask_medikamente = df_medikamente['id'].isin(doppelte_ids) & (df_medikamente['abfrage'] == 'tirzepatide')
df_medikamente = df_medikamente.drop(df_medikamente[mask_medikamente].index)

# ---------- Kontrolle ----------
print('Berichte:', df_berichte.shape)             # (Zeilen, Spalten)
print('Nebenwirkungen:', df_nebenwirkungen.shape)
print('Medikamente:', df_medikamente.shape)
print('Doppelte IDs:', df_berichte['id'].duplicated().sum())   # sollte 0 sein

In [ ]:
# ---------- Spalte abfrage entfernen ----------
df_berichte = df_berichte.drop(columns='abfrage')             # Hilfsspalte entfernen
df_nebenwirkungen = df_nebenwirkungen.drop(columns='abfrage')
df_medikamente = df_medikamente.drop(columns='abfrage')

# ---------- Kontrolle ----------
print('Berichte:', df_berichte.shape)             
print('Nebenwirkungen:', df_nebenwirkungen.shape) 
print('Medikamente:', df_medikamente.shape)       

# ---------- Speichern ----------
df_berichte.to_csv('data/processed/berichte.csv', index=False)
df_nebenwirkungen.to_csv('data/processed/nebenwirkungen.csv', index=False)
df_medikamente.to_csv('data/processed/medikamente.csv', index=False)

In [ ]:
df_berichte.head()

In [ ]:
df_berichte.dtypes

In [ ]:
df_berichte['datum'] = pd.to_datetime(df_berichte['datum'], format='%Y%m%d')   # Spalte datum in Datumsformat umwandeln

int_columns = ['schwerwiegend', 'geschlecht', 'meldende', 'berichtsart', 'alter_einheit']   # Spalten, die in Ganzzahldatentyp umgewandelt werden sollen
for col in int_columns:
    df_berichte[col] = df_berichte.loc[:, col].astype('Int64')   # Spalte col in Ganzzahldatentyp umwandeln

df_berichte['alter'] = df_berichte.loc[:, 'alter'].astype('float')   # Spalte alter in Fließkommazahl umwandeln

In [ ]:
df_berichte.dtypes

In [ ]:
df_berichte.describe()

In [ ]:
faktor_in_jahre = {800: 10, 801: 1, 802: 1/12, 803: 1/52, 804: 1/365, 805: 1/8760}   # Code → Faktor, um auf Jahre zu kommen

df_berichte['alter'] = df_berichte['alter'] * df_berichte['alter_einheit'].map(faktor_in_jahre)   # alles in Jahre umrechnen
df_berichte = df_berichte.drop(columns='alter_einheit')   # Einheit wird nicht mehr gebraucht, jetzt ist alles in Jahren
df_berichte = df_berichte.rename(columns={'alter': 'alter_jahre'})

In [ ]:
df_berichte['alter_jahre'] = pd.to_numeric(df_berichte['alter_jahre'], errors='coerce')   # sicherstellen, dass es eine Zahl ist
df_berichte['alter_jahre'] = (df_berichte['alter_jahre'] // 1).astype('Int64')           # Nachkommastellen abschneiden, dann ganze Zahl

In [ ]:
df_berichte.dtypes

In [ ]:
df_berichte['alter_jahre'].describe()  # Überblick, inklusive des Ausreißers mit 912

In [ ]:
(df_berichte['alter_jahre'] == 0).sum()   # Anzahl der Berichte mit Alter 0

In [ ]:
uebersetzungen = {   # Dictionary mit einem Dictionary pro Spalte: Code → Klartext
    'schwerwiegend': {1: 'ja', 2: 'nein'},
    'geschlecht': {0: 'unbekannt', 1: 'männlich', 2: 'weiblich'},
    'meldende': {1: 'Arzt/Ärztin', 2: 'Apotheker:in', 3: 'andere Fachkraft', 4: 'Anwalt/Anwältin', 5: 'Verbraucher:in'},
    'berichtsart': {1: 'Spontanmeldung', 2: 'Studie', 3: 'andere', 4: 'unbekannt'},
}

for spalte in uebersetzungen:   # für jede Spalte im Dictionary ...
    df_berichte[spalte] = df_berichte[spalte].map(uebersetzungen[spalte]).astype('category')   # Codes übersetzen, dann als Kategorie speichern

In [ ]:
df_berichte.isna().sum()

In [ ]:
cat_columns = ['schwerwiegend', 'geschlecht', 'meldende', 'berichtsart', 'alter_jahre']
for col in cat_columns:
    print(df_berichte.loc[:,col].unique())

In [ ]:
df_berichte.duplicated().sum()

In [ ]:
df_berichte.dtypes

In [ ]:
df_berichte['id'].duplicated().sum()   # sollte 0 sein, sonst stimmt was nicht

In [ ]:
df_berichte.to_csv('data/processed/berichte_bereinigt.csv', index=False)   # bereinigte Version unter neuem Namen speichern

In [ ]:
df_medikamente.head()

In [ ]:
df_medikamente.dtypes

In [ ]:
# ---------- Spalte rolle übersetzen ----------
rolle_uebersetzung = {1: 'verdächtig', 2: 'Begleitmedikation', 3: 'Wechselwirkung'}
df_medikamente['rolle'] = pd.to_numeric(df_medikamente['rolle'], errors='coerce').map(rolle_uebersetzung).astype('category')

In [ ]:
# ---------- Untersuchung der Spalte rolle ----------
pd.crosstab(index=df_medikamente.loc[:, 'rolle'], columns='count', normalize='columns') # Wichtig: Anteile an Einträgen nicht an Berichten

In [ ]:
df_medikamente['rolle'].value_counts(dropna=False, normalize=True)

In [ ]:
df_medikamente.head()

In [ ]:
# Wirkstoff: Wie viele Werte sind NICHT komplett großgeschrieben?
(df_medikamente['wirkstoff'].str.isupper() == False).sum()

In [ ]:
# Medikament: Wie viele Werte sind NICHT komplett großgeschrieben?
(df_medikamente['medikament'].str.isupper() == False).sum()

In [ ]:
# Anwendung: Wie viele Werte sind NICHT in der Form "Erster Buchstabe groß, Rest klein"?
(df_medikamente['anwendung'] != df_medikamente['anwendung'].str.capitalize()).sum()

In [ ]:
# Schreibe alle Spalten in Großbuchstaben, damit die Einträge einheitlich sind
for col in ['medikament', 'wirkstoff', 'anwendung']:
    df_medikamente[col] = df_medikamente[col].str.upper().str.strip()

In [ ]:
print((df_medikamente['wirkstoff'].str.isupper() == False).sum())   # sollte 0 sein
print((df_medikamente['medikament'].str.isupper() == False).sum())   # sollte 0 sein
print((df_medikamente['anwendung'].str.isupper() == False).sum())   # sollte 0 sein

In [ ]:
# ---------- Kontrolle auf fehlende Werte ------
print(df_medikamente['wirkstoff'].isna().sum())    # sollte 4568 sein
print(df_medikamente['anwendung'].isna().sum())    # sollte 91538 sein

In [ ]:
# ---------- Wieoft steht ein Wirkstoff mehrfach in einem Bericht? ----------
df_medikamente.duplicated(subset=['id', 'wirkstoff']).sum()   # wie oft kommt derselbe Wirkstoff im selben Bericht nochmal vor?

In [ ]:
# ---------- Nähere Untersuchung der Duplikate ----------
mask_doppelt = df_medikamente.duplicated(subset=['id', 'wirkstoff'], keep=False)   # alle Zeilen, die zu einer Wiederholung gehören
beispiel_id = df_medikamente[mask_doppelt]['id'].iloc[0]   # die erste betroffene ID nehmen
df_medikamente[df_medikamente['id'] == beispiel_id]   # alle Medikamente dieses Berichts anzeigen

In [ ]:
# ---------- Entfernen der Duplikate, welche vermutlich wegen mehrerer Dosierungsangaben pro Medikament entstanden sind ----------
print('vorher:', df_medikamente.shape)
df_medikamente = df_medikamente.drop_duplicates()   # Nur Zeilen löschen, die in ALLEN Spalten gleich sind
print('nachher:', df_medikamente.shape)

In [ ]:
# ---------- Gibt es noch weiter Duplikate? ----------
print(df_medikamente.duplicated(subset=['id', 'wirkstoff', 'rolle']).sum())
print(df_medikamente.duplicated(subset=['id', 'medikament', 'rolle']).sum())

In [ ]:
mask_rest = df_medikamente.duplicated(subset=['id', 'medikament', 'rolle'], keep=False)   # alle Zeilen, die zu so einem Paar gehören
df_medikamente[mask_rest].sort_values(['id', 'medikament']).head(10)   # sortiert, damit die Paare untereinander stehen

In [ ]:
# Duplikate, welche vermutlich wegen unterschiedlicher Anwendungsangaben entstanden sind, dürfen drinbleiben, sollten aber in der weiteren Analyse berücksichtigt werden

In [ ]:
print(df_medikamente.duplicated(subset=['id', 'wirkstoff', 'rolle', 'anwendung']).sum())
print(df_medikamente.duplicated(subset=['id', 'medikament', 'rolle', 'anwendung']).sum())

In [ ]:
mask_name = df_medikamente.duplicated(subset=['id', 'wirkstoff', 'rolle', 'anwendung'], keep=False)   # alle Zeilen, die zu so einem Paar gehören
df_medikamente[mask_name].sort_values(['id', 'wirkstoff']).head(10)   # sortiert, damit die Paare untereinander stehen

In [ ]:
# Kann verschiedene Grunde für Duplikate geben: 
    # a) unterschiedliche Medikamente mit gleichen Wirkstoff (z.B. Ozempic/Wegovy/Rybelsus, Patient ist z.B. von Spritze (Ozempic) auf Tabletten (Rybelsus) umgestiegen)
    # b) Unechte Doppler wegen fehlender Wirkstoffangabe

In [ ]:
df_medikamente[mask_name]['medikament'].value_counts().head(15)   # welche Namen tauchen in solchen Paaren am häufigsten auf?

In [ ]:
mask_eins = df_medikamente.duplicated(subset=['id', 'medikament', 'rolle', 'anwendung'], keep=False)
df_medikamente[mask_eins]

In [ ]:
# Anzahl der Einträge ohne Wirkstoff
print(df_medikamente['wirkstoff'].isna().sum())
df_medikamente[df_medikamente['wirkstoff'].isna()]['medikament'].value_counts().head(15)

In [ ]:
# ---------- Entfernen der Einträge ohne Wirkstoff ------
df_medikamente = df_medikamente.dropna(subset=['wirkstoff'])   # Zeilen ohne Wirkstoff entfernen
print(df_medikamente.shape)   # sollte 248435 − 4516 = 243919 Zeilen haben

In [ ]:
print(df_medikamente.duplicated(subset=['id', 'wirkstoff', 'rolle', 'anwendung']).sum())
print(df_medikamente.duplicated(subset=['id', 'medikament', 'rolle', 'anwendung']).sum())

In [ ]:
df_medikamente.to_csv('data/processed/medikamente_bereinigt.csv', index=False)   # bereinigte Medikamenten-Tabelle unter neuem Namen speichern

In [ ]:
df_nebenwirkungen.shape

In [ ]:
df_nebenwirkungen.head()

In [ ]:
df_nebenwirkungen.duplicated().sum()   # wie oft kommt eine Nebenwirkung mehr als einmal vor?

In [ ]:
mask_doppelt = df_nebenwirkungen.duplicated(keep=False)   # alle Zeilen, die zu einem Doppler gehören
df_nebenwirkungen[mask_doppelt].sort_values(['id', 'nebenwirkung']).head(10)

In [ ]:
print(df_nebenwirkungen['nebenwirkung'].nunique())                      # Anzahl verschiedener Nebenwirkungen, so wie sie sind
print(df_nebenwirkungen['nebenwirkung'].str.upper().str.strip().nunique())   # Anzahl, wenn man Groß/klein und Leerzeichen ignoriert

In [ ]:
# ---------- Doppler entfernen ----------
# 1.710 von 226.536 Einträgen (0,75 %) sind identische Doppler: gleiche Nebenwirkung im selben Bericht.
# Wahrscheinliche Herkunft:
#   - Verschiedene Formulierungen im Originalbericht wurden auf denselben MedDRA-Fachbegriff abgebildet
#     (z. B. "mir war schlecht" und "Übelkeit" -> beides NAUSEA).
#   - Die Einträge unterscheiden sich nur in Feldern, die nicht übernommen wurden (z. B. Ausgang der Nebenwirkung).
# Warum entfernen: Für die Analyse zählt nur, OB eine Nebenwirkung in einem Bericht vorkommt, nicht wie oft.
# Ohne Entfernen würde ein Bericht mit zweimal derselben Nebenwirkung doppelt gezählt.

print('vorher:', df_nebenwirkungen.shape)
df_nebenwirkungen = df_nebenwirkungen.drop_duplicates()   # jede Nebenwirkung pro Bericht nur einmal behalten
print('nachher:', df_nebenwirkungen.shape)                # erwartet: 224826 Zeilen (226536 − 1710)

In [ ]:
df_nebenwirkungen['nebenwirkung'].isna().sum() # wie viele Einträge haben keine Nebenwirkung?

In [ ]:
df_nebenwirkungen['id'].isna().sum() # wie viele Einträge haben keine id?

In [ ]:
df_nebenwirkungen.dtypes

In [ ]:
df_nebenwirkungen.to_csv('data/processed/nebenwirkungen_bereinigt.csv', index=False)